# einops-rearrange-flatten composite — cx4: flatten a CHW image then tile across a new batch axis

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-rearrange-flatten`, `einops-repeat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-rearrange-flatten"
DD_ATOM_IDS = ["einops-rearrange-flatten", "einops-repeat"]
DD_SUBTOPICS = ["Einops: Rearrange-as-flatten", "Einops: Repeat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

`einops-rearrange-flatten` collapses named axes via parens: `'c h w -> (c h w)'` flattens a 3-D feature map into a 1-D vector. `einops-repeat` then inserts a new named axis: `'features -> b features', b=B` tiles the same flat vector across a new batch dim.

**Why this pattern shows up.** When you have a single template image (e.g. a CNN's input mean or a positional embedding) and need to broadcast it as a flat feature vector across a whole training batch, you flatten first (so downstream Linear layers see `(B, D)`) and then `repeat` to insert `B`.

**Order matters.** Flatten first, repeat second — repeating BEFORE flatten gives a `(B, C, H, W)` tensor which the Linear head can't consume without another flatten.

### Composite Exercise — flatten a CHW image then tile across a new batch axis

**Atoms exercised together**: `einops-rearrange-flatten`, `einops-repeat`

Build `cx4_flatten_then_tile(x, batch)` that takes a 3-D image-like tensor `x` of shape `(C, H, W)` and tiles it across `batch` rows of a new batch axis, producing shape `(batch, C*H*W)`.

Constraints:
- Use ONE `rearrange` to flatten `(C, H, W) -> (C*H*W,)`.
- Use ONE `repeat` to add the batch axis: `'features -> b features'`.
- The flat row at index `i` of the output must equal `x.reshape(C*H*W)` for every `i`.
- No `.unsqueeze().expand()`, no `torch.stack`, no manual loops.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx4_flatten_then_tile(x, batch):
    """Flatten (C,H,W) then repeat to (batch, C*H*W)."""
    raise NotImplementedError()

def _test_cx4():
    # --- (a) basic shape ---
    x = t.arange(2 * 3 * 4).reshape(2, 3, 4).float()
    out = cx4_flatten_then_tile(x, batch=5)
    assert out.shape == (5, 24), f'expected (5, 24), got {tuple(out.shape)}'

    # --- (b) every row equals the flat reference vector ---
    flat_ref = x.reshape(24)
    for b in range(5):
        assert t.equal(out[b], flat_ref), f'row {b} differs from flat reference'

    # --- (c) inner-loop order is (c h w), matches reshape ---
    x2 = t.tensor([[[0., 1.], [2., 3.]],
                   [[4., 5.], [6., 7.]]])  # (C=2, H=2, W=2)
    out2 = cx4_flatten_then_tile(x2, batch=3)
    assert out2.shape == (3, 8)
    assert t.equal(out2[0], t.tensor([0., 1., 2., 3., 4., 5., 6., 7.]))

    # --- (d) batch=1 still works ---
    out3 = cx4_flatten_then_tile(x, batch=1)
    assert out3.shape == (1, 24)
    assert t.equal(out3[0], flat_ref)
    _dd_passed.add('cx4')

_test_cx4()

<details><summary>Show solution — cx4</summary>

```python
def cx4_flatten_then_tile(x, batch):
    flat = rearrange(x, 'c h w -> (c h w)')        # flatten atom
    return repeat(flat, 'd -> b d', b=batch)        # repeat atom adds B
```

Two atoms, two lines. The flatten is `'c h w -> (c h w)'` (grouped axis collapses to a single 1-D dim); the tile is `'d -> b d', b=B` (new axis introduced by repeat). Doing the steps in the wrong order (tile-then-flatten) gives `(batch, C*H*W)` but only via a second flatten step — and a stricter test would catch the intermediate shape.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx4'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx4',
        'subtopics': ["Einops: Rearrange-as-flatten", "Einops: Repeat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()